
# CRIM Intervals:  Melodic and Harmonic Corpus Search

### What You Can Do with this Notebook:

* Search A Corpus for Melodic and Harmonic nGrams

### A. Import Intervals and Other Code


In [1]:
import intervals
from intervals import * 
from intervals import main_objs
import intervals.visualizations as viz
import pandas as pd
import re
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact
from pandas.io.json import json_normalize
from pyvis.network import Network
from IPython.display import display
import requests
import os
import glob as glob


MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)
else:
    print(MUSDIR, "folder already exists.")

ModuleNotFoundError: No module named 'intervals'

### Ricerca preimpostata (corpus preimpostato; kind='d', n=4, combineUnisons=True; soggetti preimpostati)

In [ ]:
# ESEGUI LA CELLA e seleziona il soggetto
corpus = CorpusBase(['https://crimproject.org/mei/CRIM_Model_0019.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_1.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_2.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_3.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_4.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_5.mei'])

soggetti = ['-3, 3, 2, -2', '-3, 2, 2, -2', '-3, 3, -2, -2', '-2, 2, 3, -2']

func1 = ImportedPiece.notes
notes_df = corpus.batch(func=func1, kwargs={'combineUnisons': True}, metadata=False)
func2 = ImportedPiece.melodic
melodic_df = corpus.batch(func=func2, kwargs={'kind': 'd', 'end': False, 'df': notes_df}, metadata=False)
func3 = ImportedPiece.ngrams
ngrams_df = corpus.batch(func=func3, kwargs={'n': 4, 'df': melodic_df}, metadata=False)
func4 = ImportedPiece.detailIndex
list_of_detail_index = corpus.batch(func=func4, kwargs={'offset': False,'df': ngrams_df}, metadata=True)

mel_corpus = pd.concat(list_of_detail_index)
comp = mel_corpus.pop("Composer")
mel_corpus['Composer'] = comp
title = mel_corpus.pop("Title")
mel_corpus["Title"] = title
mel_corpus = mel_corpus.fillna('-')

def _convertTuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out

@interact
def mel_ngram_search(soggetto=soggetti, df = fixed(mel_corpus)):
    df_no_tuple = df.applymap(_convertTuple)
    df_no_tuple.pop("Composer")
    df_no_tuple.pop("Title")
    df_no_tuple.insert(0, "Composer", df["Composer"])
    df_no_tuple.insert(1, "Title", df["Title"])
    filtered_ngrams = df_no_tuple[df_no_tuple.apply(lambda x: x.astype(str).str.contains(soggetto).any(), axis=1)].copy()
    
    pd.set_option('max_columns', None)
    return filtered_ngrams.fillna("-").reset_index().applymap(str).style.applymap(lambda x: "background: #ccebc4" if re.search(soggetto, x) else "")


### Imposta i tuoi parametri

In [ ]:
# componi il corpus
corpus = CorpusBase(['https://crimproject.org/mei/CRIM_Model_0019.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_1.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_2.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_3.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_4.mei',
                     'https://crimproject.org/mei/CRIM_Mass_0019_5.mei'])

In [ ]:
# formato dei motivi
combineUnisons = True
kind = 'd'
n = 5

In [ ]:
# ESEGUI LA CELLA e scrivi la tua ricerca
func1 = ImportedPiece.notes
notes_df = corpus.batch(func=func1, kwargs={'combineUnisons': combineUnisons}, metadata=False)
func2 = ImportedPiece.melodic
melodic_df = corpus.batch(func=func2, kwargs={'kind': kind, 'end': False, 'df': notes_df}, metadata=False)
func3 = ImportedPiece.ngrams
ngrams_df = corpus.batch(func=func3, kwargs={'n': n, 'df': melodic_df}, metadata=False)
func4 = ImportedPiece.detailIndex
list_of_detail_index = corpus.batch(func=func4, kwargs={'offset': False,'df': ngrams_df}, metadata=True)

mel_corpus = pd.concat(list_of_detail_index)
comp = mel_corpus.pop("Composer")
mel_corpus['Composer'] = comp
title = mel_corpus.pop("Title")
mel_corpus["Title"] = title
mel_corpus = mel_corpus.fillna('-')

def _convertTuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out

@interact
def mel_ngram_search(my_search="", df = fixed(mel_corpus)):
    df_no_tuple = df.applymap(_convertTuple)
    df_no_tuple.pop("Composer")
    df_no_tuple.pop("Title")
    df_no_tuple.insert(0, "Composer", df["Composer"])
    df_no_tuple.insert(1, "Title", df["Title"])
    filtered_ngrams = df_no_tuple[df_no_tuple.apply(lambda x: x.astype(str).str.contains(my_search).any(), axis=1)].copy()
    
    pd.set_option('max_columns', None)
    return filtered_ngrams.fillna("-").reset_index().applymap(str).style.applymap(lambda x: "background: #ccebc4" if re.search(my_search, x) else "")

### Controlla la musica

In [ ]:
piece_list = ['CRIM_Model_0019.mei',
              'CRIM_Mass_0019_1.mei',
              'CRIM_Mass_0019_2.mei',
              'CRIM_Mass_0019_3.mei',
              'CRIM_Mass_0019_4.mei',
              'CRIM_Mass_0019_5.mei']
prefix = 'https://crimproject.org/mei/' 
mei_file = piece_list[0] # 0=model 1-5=mass movements (e. g. 2=Gloria)
url = prefix + mei_file
piece = importScore(url)
print(piece.metadata)
piece.verovioPrintExample(39, 42) # start measure, end measure